# MaSIF-ligand: classify a binding pocket

Run MaSIF-ligand on one prepared binding pocket.

This notebook previews the configuration before running your data. Running every cell with the default settings does not start a prediction.


## 1. Before you start

Install MaSIF using the [README installation instructions](../README.md#installation). Use a notebook kernel from your `masif_torch` environment, and obtain the checkpoint listed in the [weights guide](../weights/README.md).

Follow [Open a tutorial](README.md#open-a-tutorial) to install Jupyter and register the `masif_torch` kernel. Confirm the selected notebook kernel uses the same Python as your MaSIF installation.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Locate the checkout whether this notebook starts in tutorials/ or the project root.
start = Path.cwd().resolve()
project_root = next(
    (path for path in (start, *start.parents)
     if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch checkout.")
os.chdir(project_root)
print("Project:", project_root)
print("Notebook Python:", sys.executable)


## 2. Prepare your input

Provide a prepared **pocket NPZ** containing the surface patches for the pocket you want to classify. The single-pocket command below does not extract a pocket from a raw PDB file.

A directory of converted original TFRecord **structure packets** is used for dataset evaluation or training, rather than as a single-pocket input. See [the usage manual](../USAGE.md) for TFRecord conversion and surface/ligand-coordinate packing.

For raw protein–ligand surface preparation, use the [preparation tutorial](prepare_ligand.ipynb).


## 3. Choose paths and settings

Replace `input_path` with your prepared packet. Choose an output path for this run. Keep `device = "cpu"` for a CPU run, or select `"cuda"` after installing a compatible GPU runtime.

Leave `RUN_TASK = False` while checking the configuration. Set it to `True` only when the input and checkpoint are ready.


In [ ]:
checkpoint = Path("weights/ligand/pocket_classification.pt")
input_path = Path("data/ligand/example.npz")
output_path = Path("results/ligand/example.npy")
device = "cpu"
batch_size = 100
RUN_TASK = False

command = [
    sys.executable, "-m", "masif", "ligand",
    "--config", "examples/ligand.yaml",
    f"checkpoint={checkpoint}",
    f"input={input_path}",
    f"output={output_path}",
    f"device={device}",
    f"batch_size={batch_size}",
]


## 4. Preview the configuration

This step checks the command and displays your settings without reading the input or loading a checkpoint. Review the paths and device before continuing.


In [ ]:
print(shlex.join(command))
preview = subprocess.run(command + ["--cfg", "job"], check=True, capture_output=True, text=True)
print(preview.stdout)


## 5. Run your prepared input

After updating the settings, set `RUN_TASK = True` and rerun the settings, preview and run cells. The run cell checks that the input and checkpoint exist before starting.


In [ ]:
if RUN_TASK:
    for required_path in (input_path, checkpoint):
        if not required_path.is_file():
            raise FileNotFoundError(f"Required file not found: {required_path}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(command, check=True)
    print("Saved:", output_path)
else:
    print("Preview only. Set RUN_TASK = True after preparing your files.")


## 6. Next steps

To evaluate a directory of converted structure packets instead, run:

```bash
masif evaluate-ligand --checkpoint weights/ligand/pocket_classification.pt \
  --input data/ligand/test --output results/ligand/test --device cpu
```

For training with separate training and validation splits, continue with the [training tutorial](training.ipynb).

See the [usage manual](../USAGE.md) for other commands and configuration examples.
